> **Note:** originally developed and run in Google Colab. Google-Drive mounting cells were removed, Drive paths were rewritten as relative paths and outputs were stripped (figures of the original run are in `assets/`). The paths/cleanup were **not re-executed** after this change, so expect to adjust paths.

In [ ]:

!pip install -e ..

In [ ]:
import geobio as gb

avistajes = "data/ebd_AR_slathr2_unv_smp_relApr-2025.txt"
checklist = "data/ebd_AR_slathr2_unv_smp_relApr-2025_sampling.txt"

# **Demo**

In [ ]:
merged = gb.build_presence_from_checklists(
    sightings_path = avistajes,
    checklist_path = checklist)

In [ ]:
print(f'El tamaño de los datos es de {merged.shape}')
merged.head()

In [ ]:
merged_filtered = gb.filter_data(
    df = merged,
    max_distance_km = 5,
    max_observers = 10,
    max_duration_min= 300,
    allowed_protocols = ['Stationary', 'Traveling'],
    drop_duplicates= True
)

In [ ]:
print(f'El tamaño de los datos filtrados es de {merged_filtered.shape}')
merged_filtered.head()

In [ ]:
filtro_radio = gb.plot_checklists_radio_static(
    merged= merged_filtered,
    umbral_km=100.0,
    #base_gdf=ecorregion_gdf  # opcional
)

In [ ]:
filtro_radio.head()

In [ ]:
filtered_eco, info = gb.filtrar_checklists_por_region_fast(
    merged_filtered,
    region_path="data/Ecoregiones2.shp",
)

gb.plot_region_checklists_from_fast(
    filtered_eco,
    region_path="data/Ecoregiones2.shp",
    merged_original=merged_filtered,             # opcional, para mostrar los “no kept”
    titulo="Presencias (azul) y ausencias (rojo) dentro de la Ecoregión Ucayali moist forests",
    color_presencia="royalblue",
    color_ausencia="orangered",
    color_contexto="lightgray"
)


In [ ]:
filtered_eco.head()

In [ ]:
rasters = {
    'Temperatura': 'data/CHELSA_bio1_1981-2010_V.2.1.tif',
     # Modificamos las rutas y los nombres de las columnas nuevas como sea necesario.
    'Precipitacion': 'data/CHELSA_bio12_1981-2010_V.2.1.tif',
     # Podemos adicionar cuantos rasters queramos utilizar.
}

# La función genera un nuevo archivo csv con las columnas de los datos climáticos que hayamos extraído.
datos_tratados = gb.extraer_variables_especificas(filtered_eco, rasters, archivo_salida='datos_con_variables.csv')
datos_tratados

In [ ]:
datos_tratados['OBSERVATION DATE'].isna().sum()

In [ ]:
merged_con_fechas = gb.separar_fechas(datos_tratados)

In [ ]:
# 1. Agregar estación
merged_con_estacion = gb.add_season(merged_con_fechas)


In [ ]:
# 2. Filtrar solo verano e invierno
merged_filtrado = gb.filter_seasons(merged_con_estacion, keep=("Verano", "Invierno"))

In [ ]:
# Re-run the function call after fixing the separator issue
html_path = gb.crear_mapa_interactivo(avistajes, sep='\t')

In [ ]:
ruta_geojson = "data/data.geojson"  # Modificar la ruta a nuestro gson.
df = pd.read_csv(avistajes, sep= '\t')

gdf_seg = gb.filtrar_por_poligonos(
        datos= df,
        ruta_poligono=ruta_geojson,
        columna_especie='SCIENTIFIC NAME',  # <-- ajusta si tu CSV usa otro nombre
        archivo_salida='datos_segmentados.csv'
    )

gdf_seg # Vemos la tabla con los segmentos

In [ ]:
rasters = {
    'Temperatura': 'data/CHELSA_bio1_1981-2010_V.2.1.tif',
     # Modificamos las rutas y los nombres de las columnas nuevas como sea necesario.
    'Precipitacion': 'data/CHELSA_bio12_1981-2010_V.2.1.tif',
     # Podemos adicionar cuantos rasters queramos utilizar.
}

# La función genera un nuevo archivo csv con las columnas de los datos climáticos que hayamos extraído.
datos_tratados_seg = gb.extraer_variables_especificas(gdf_seg, rasters, archivo_salida='datos_con_variables.csv')

In [ ]:
merged_con_fechas_seg = gb.separar_fechas(datos_tratados_seg)
merged_con_estacion_seg = gb.add_season(merged_con_fechas_seg)

In [ ]:
gb.plot_mirror_presence_by_station(
    df = merged_con_estacion, # Use the already filtered DataFrame
    station_col="Estación",
    stations=["Verano", "Invierno"],                    # lista de estaciones a incluir; si None, usa todas las presentes
    bin_size = 1.0,
    title = 'Esfuerzo de muestreo (todas las checklist)',
    figsize=(12, 8))

In [ ]:
positive_presences = merged_con_estacion[merged_con_estacion['Presencia'] == 'Positiva']

gb.plot_latitude_density(positive_presences)


In [ ]:
gb.plot_temperature_density(positive_presences)

In [ ]:
gb.plot_precipitation_density(positive_presences)

In [ ]:
gb.plot_precipitation_by_segment(merged_con_estacion_seg)

In [ ]:
gb.plot_latitude_by_segment(merged_con_estacion_seg)

In [ ]:
gb.plot_monthly_frequencies(merged_con_estacion_seg)